#recreate correlation plot


In [1]:
from __future__ import annotations

import argparse
import glob
import os
import sys

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import overcomplete_learning.plotting as ol_plot
import overcomplete_learning.data       as ol_data
# Table 6.1 metrics, in target-grouped order: (csv_column, display_label, target)

#!/usr/bin/env python3
"""
error_metric_correlation.py
===========================

Pearson-correlation matrix of the error metrics from Table 6.1
(``tab:metrics_summary``), computed across a folder of simulation-result CSVs and
rendered as an annotated heatmap grid.

The six metrics (grouped by their estimation target) are:

    target X-hat :  X_err              -- reconstruction error
    target A-hat :  A_err              -- column-scaled MSE (F_err)
                    A_err_angle_mean   -- mean column-wise angular error
                    A_err_angle_max    -- maximal angular error
    target S-hat :  S_err_MCC          -- source mean correlation coefficient
                    S_err_MSE          -- weighted reconstruction error

Note on direction: five metrics are "lower = better" (best score 0) but
``S_err_MCC`` is "higher = better" (best score 1).  By default it is aligned to
the error direction (``1 - MCC``) so that metrics sharing a target correlate
*positively* and the grid reads cleanly; pass ``--raw-mcc`` to keep it as-is.

Usage
-----
    python error_metric_correlation.py [FOLDER] [--out FIG.png] [options]

    FOLDER              directory of result CSVs (default: ./results_final)
    --out PATH          output image (default: <folder>/error_metric_correlation.png)
    --recursive         also search sub-directories for *.csv
    --raw-mcc           use raw S_err_MCC instead of 1 - S_err_MCC
    --method NAME       keep only rows whose 'method' column equals NAME
    --converged-only    keep only rows with converged == True
    --pattern GLOB      filename glob (default: *.csv)
"""



METRICS = [
    ("X_err",            ol_plot.metric_label("X_err"),       "X"),   # reconstruction error
    ("A_err",            ol_plot.metric_label("A_err"),       "A"),   # column-scaled MSE (F_err)
    ("A_err_angle_mean", ol_plot.metric_label("A_err_angle_mean"),  "A"),   # mean column-wise angular error
    ("A_err_angle_max",  ol_plot.metric_label("A_err_angle_max"),   "A"),   # maximal angular error
    ("S_err_MCC",        ol_plot.metric_label("S_err_MCC"),       "S"),   # source mean correlation coeff.
    ("S_err_MSE",        ol_plot.metric_label("S_err_MSE"),       "S"),   # weighted reconstruction error
]
TARGET_COLORS = {"X": "#4C72B0", "A": "#DD8452", "S": "#55A868"}


def load_folder(folder, pattern="*.csv", recursive=False):
    """Load and concatenate every matching CSV in *folder*."""
    if recursive:
        files = glob.glob(os.path.join(folder, "**", pattern), recursive=True)
    else:
        files = glob.glob(os.path.join(folder, pattern))
    files = sorted(files)
    if not files:
        raise FileNotFoundError(f"No files matching {pattern!r} in {folder!r}")
    frames = []
    for f in files:
        try:
            frames.append(pd.read_csv(f))
        except Exception as e:                      # skip unreadable/empty files
            print(f"  ! skipping {os.path.basename(f)}: {e}", file=sys.stderr)
    df = pd.concat(frames, ignore_index=True)
    print(f"Loaded {len(files)} files, {len(df):,} rows from {folder}")
    return df


def build_metric_frame(df, align_mcc=True, method=None, converged_only=False):
    """Filter rows and return a numeric frame of the available Table-6.1 metrics."""
    if method is not None:
        if "method" not in df.columns:
            raise KeyError("'method' column not present; cannot filter by method")
        df = df[df["method"] == method]
    if converged_only and "converged" in df.columns:
        df = df[df["converged"].astype(str).str.lower().isin(["true", "1"])]

    present = [(c, lab, tgt) for c, lab, tgt in METRICS if c in df.columns]
    missing = [c for c, _, _ in METRICS if c not in df.columns]
    if missing:
        print(f"  (columns not found, skipped: {', '.join(missing)})")
    if len(present) < 2:
        raise ValueError("Fewer than two Table-6.1 metric columns found in the data")

    cols, labels, targets = [], [], []
    out = {}
    for c, lab, tgt in present:
        vals = pd.to_numeric(df[c], errors="coerce").replace([np.inf, -np.inf], np.nan)
        if align_mcc and c == "S_err_MCC":
            vals = 1.0 - vals
            lab = f"1 - {ol_plot.metric_label('S_err_MCC')}"
        out[lab] = vals.to_numpy()
        cols.append(c); labels.append(lab); targets.append(tgt)
    frame = pd.DataFrame(out)
    return frame, labels, targets


def plot_grid(corr, labels, targets, title, out_path):
    """Render the correlation matrix as an annotated heatmap grid."""
    n = len(labels)
    fig, ax = plt.subplots(figsize=(1.15 * n + 2.5, 1.15 * n + 2.0))
    im = ax.imshow(corr, vmin=-1, vmax=1, cmap="RdBu_r", aspect="equal")

    ax.set_xticks(range(n)); ax.set_yticks(range(n))
    ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=9)
    ax.set_yticklabels(labels, fontsize=9)
    # colour tick labels by target group
    for tick, tgt in zip(ax.get_xticklabels(), targets):
        tick.set_color(TARGET_COLORS.get(tgt, "black"))
    for tick, tgt in zip(ax.get_yticklabels(), targets):
        tick.set_color(TARGET_COLORS.get(tgt, "black"))

    # annotate each cell with the correlation value
    for i in range(n):
        for j in range(n):
            v = corr[i, j]
            if np.isnan(v):
                txt = "—"
            else:
                txt = f"{v:.2f}"
            ax.text(j, i, txt, ha="center", va="center", fontsize=8,
                    color="white" if abs(v) > 0.6 else "black")

    # dividers between target groups
    bounds = [k for k in range(1, n) if targets[k] != targets[k - 1]]
    for b in bounds:
        ax.axhline(b - 0.5, color="black", lw=1.4)
        ax.axvline(b - 0.5, color="black", lw=1.4)

    ax.set_title(title, fontsize=11, pad=12)
    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label("Pearson correlation", fontsize=9)
    fig.tight_layout()
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved figure -> {out_path}")


def main(argv=None):
    p = argparse.ArgumentParser(description=__doc__,
                                formatter_class=argparse.RawDescriptionHelpFormatter)
    here = '../scripts/'
    p.add_argument("folder", nargs="?", default=os.path.join(here, "results_final"),
                   help="folder of result CSVs (default: ./results_final)")
    p.add_argument("--out", default=None, help="output image path")
    p.add_argument("--recursive", action="store_true", help="recurse into sub-folders")
    p.add_argument("--raw-mcc", action="store_true", help="keep raw S_err_MCC")
    p.add_argument("--method", default=None, help="filter to a single method")
    p.add_argument("--converged-only", action="store_true", help="keep converged rows only")
    p.add_argument("--pattern", default="*.csv", help="filename glob")
    args = p.parse_args(argv)

    df = load_folder(args.folder, pattern=args.pattern, recursive=args.recursive)
    frame, labels, targets = build_metric_frame(
        df, align_mcc=not args.raw_mcc, method=args.method,
        converged_only=args.converged_only)

    corr = frame.corr(method="pearson")            # pairwise-complete Pearson
    n_used = int(frame.dropna().shape[0])
    print("\nPearson correlation matrix:\n")
    print(corr.round(3).to_string())

    out_path = args.out or os.path.join(args.folder, "error_metric_correlation.png")
    csv_path = os.path.splitext(out_path)[0] + ".csv"
    corr.to_csv(csv_path)
    print(f"\nSaved matrix  -> {csv_path}")

    title = (f"Error-metric Pearson correlation")
             #f"{os.path.basename(os.path.normpath(args.folder))}  ")
             #f"{len(df):,} rows ({n_used:,} complete)"
             #+ (f" | method={args.method}" if args.method else ""))
    plot_grid(corr.to_numpy(), labels, targets, title, out_path)


In [2]:
main(["../results", "--recursive"])


Loaded 725 files, 2,262,155 rows from ../results

Pearson correlation matrix:

                          Reconstruction MSE  Column Scaled RMSE  Mean Angular Error  Maximum Angular Error  1 - Source MCC  Source Relative L2 Error
Reconstruction MSE                     1.000               0.147               0.166                  0.102           0.147                     0.004
Column Scaled RMSE                     0.147               1.000               0.977                  0.916           0.877                     0.008
Mean Angular Error                     0.166               0.977               1.000                  0.832           0.833                     0.008
Maximum Angular Error                  0.102               0.916               0.832                  1.000           0.830                     0.012
1 - Source MCC                         0.147               0.877               0.833                  0.830           1.000                     0.012
Source Relative L2 Er